_Updated date: January 11, 2026_

# 🎓 Databricks Workshop: Data & Analytics
**For BI Organization - Operational Reporting & Analytics**

---

## 👥 Welcome, BI & Analytics Team!

This workshop is specifically designed for the **BI Organization** - analysts who do:
- 📊 **Operational Reporting** to run day-to-day business
- ✅ **Compliance Reporting** for state and federal requirements
- 📞 **Contact Center Analytics** - call data and performance metrics
- 👥 **Member Services & Open Enrollment** support
- 🔍 **Research and Insights** for business decision-making

**Your Role**: As BI analysts, you'll learn how to leverage Databricks to build operational reports and analytics that support your organization.

---

## 📚 Workshop Objectives

By the end of this workshop, you will be able to:

1. ✅ Build **Medallion Architecture** pipelines for enterprise data
2. ✅ Write **Databricks SQL** queries for operational reporting
3. ✅ Create **aggregated analytics** and business metrics
4. ✅ Perform **data quality audits** and compliance validation
5. ✅ Build **Gold layer analytics** for operational insights and reporting
6. ✅ Apply **best practices** for production pipelines
7. ✅ Optimize **query performance** using Databricks features

---

## 💼 Business Use Case: BI Operational Reporting

This workshop uses a **healthcare payer dataset** as an example that reflects your daily work:

- 📞 **Contact Center Analytics**: Call volumes, wait times, resolution rates
- ✅ **Compliance Reporting**: State and federal regulatory requirements
- 💰 **Claims Operations**: Processing metrics and tracking
- 👥 **Member Services**: Enrollment tracking and member support metrics
- 📈 **Operational Dashboards**: Day-to-day business performance KPIs

---



# Databricks Medallion Architecture for Business Intelligence


## Business Analytics & Data Modeling Concepts

### 📊 Understanding the Dataset

For this workshop, we'll use a **healthcare payer dataset** as our example. The concepts you'll learn apply universally to any business domain.

**Key Concepts:**
1. **Customer Data** → Demographics, attributes, segments
2. **Transaction Data** → Claims, purchases, interactions
3. **Reference Data** → Categories, codes, mappings
4. **Provider/Vendor Data** → Service providers, suppliers
5. **Metrics & KPIs** → Calculated business measures

### 🏗️ Data Architecture Patterns

**Medallion Architecture** is an industry-standard approach for organizing data:

- **Bronze Layer** (Raw): Data ingested as-is from source systems
- **Silver Layer** (Cleansed): Validated, deduplicated, conformed data
- **Gold Layer** (Analytics): Business-level aggregates and metrics

### 📊 Data Model Overview

For our example dataset, key tables include:
- **Members**: Member demographics and enrollment information
- **Claims**: Transaction records for claims processing
- **Diagnoses**: Diagnosis codes for classification
- **Providers**: Provider network information
- **Procedures**: Procedure details and costs

<div style="display: flex; justify-content: space-between;">
  <img src="https://user-gen-media-assets.s3.amazonaws.com/gpt4o_images/5c87faea-3e60-4f71-826d-42d04f6cdc0b.png" alt="Dimensional Model" width="400" height="350">
  <img src="https://user-gen-media-assets.s3.amazonaws.com/gpt4o_images/6826c275-d462-4c07-a978-43fe9c40f3ed.png" alt="Data Vault" width="400" height="350">
</div>

**Resources:**
- [Implementing Dimensional Modeling on Databricks](https://www.databricks.com/blog/implementing-dimensional-data-warehouse-databricks-sql-part-1)
- [Medallion Architecture](https://www.databricks.com/glossary/medallion-architecture)







# 🔄 Why Move to Databricks for BI Operational Reporting?

### Your Current Environment

You currently work with:
- **Upstream data** in SQL Server for operational reporting
- **Contact center and call data** in SQL Server
- **SQL-based reporting** for compliance, claims, member services
- **Traditional BI tools** for dashboards and reports

### Challenges with Current Infrastructure

1. **🐌 Performance Bottlenecks**
   - Large datasets slow down queries during peak hours
   - Complex reports take time to refresh
   - Limited ability to handle growing data volumes
   - Difficult to run ad-hoc analysis on historical data

2. **☁️ Cloud Migration Benefits**
   - Upstream data will be migrated to cloud
   - Downstream consumers need cloud-ready analytics
   - Scalable infrastructure for growing data needs
   - Better integration across data sources

3. **🔧 Modern Analytics Capabilities**
   - Advanced SQL features (window functions, array operations)
   - Real-time dashboards and monitoring
   - Better collaboration tools for team
   - Unified platform for all analytics needs

### Databricks Advantages for BI Organization

| **Capability** | **Impact for Your Work** |
|----------------|--------------------------|
| **Modern SQL** | Familiar SQL syntax with advanced features (CTEs, window functions) |
| **Delta Lake** | Time travel for audit trails, ACID transactions for data quality |
| **Unity Catalog** | Data governance and access control for compliance |
| **Real-time Dashboards** | Live operational metrics for contact center, claims, enrollment |
| **Collaboration** | Share queries and reports with team members |
| **Scalability** | Handle peak loads (open enrollment, year-end) without performance issues |
| **Cost Efficiency** | Pay only for compute you use, auto-scaling for demand |
| **Cloud-Ready** | Seamless integration with data cloud migration |


---


# SETUP

Just run next couple of cells for setup!

In [0]:
dbutils.widgets.text("catalog", "my_catalog", "Catalog")
dbutils.widgets.text("bronze_db", "payer_bronze", "Bronze DB")
dbutils.widgets.text("silver_db", "payer_silver", "Silver DB")
dbutils.widgets.text("gold_db", "payer_gold", "Gold DB")

catalog = dbutils.widgets.get("catalog")
bronze_db = dbutils.widgets.get("bronze_db")
silver_db = dbutils.widgets.get("silver_db")
gold_db = dbutils.widgets.get("gold_db")

path = f"/Volumes/{catalog}/{bronze_db}/payer/files/"

print(f"Catalog: {catalog}")
print(f"Bronze DB: {bronze_db}")
print(f"Silver DB: {silver_db}")
print(f"Gold DB: {gold_db}")
print(f"Path: {path}")

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {bronze_db}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {silver_db}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {gold_db}")

spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_db}.payer")

# Create the volume and folders
dbutils.fs.mkdirs(f"/Volumes/{catalog}/{bronze_db}/payer/files/claims")
dbutils.fs.mkdirs(f"/Volumes/{catalog}/{bronze_db}/payer/files/diagnosis")
dbutils.fs.mkdirs(f"/Volumes/{catalog}/{bronze_db}/payer/files/procedures")
dbutils.fs.mkdirs(f"/Volumes/{catalog}/{bronze_db}/payer/files/members")
dbutils.fs.mkdirs(f"/Volumes/{catalog}/{bronze_db}/payer/files/providers")
dbutils.fs.mkdirs(f"/Volumes/{catalog}/{bronze_db}/payer/downloads")

In [0]:
import requests
import zipfile
import io
import os
import shutil

# Define the URL of the ZIP file
url = "https://github.com/bigdatavik/databricksfirststeps/blob/6b225621c3c010a2734ab604efd79c15ec6c71b8/data/Payor_Archive.zip?raw=true"

# Download the ZIP file
response = requests.get(url)
zip_file = zipfile.ZipFile(io.BytesIO(response.content))

# Define the base path
base_path = f"/Volumes/{catalog}/{bronze_db}/payer/downloads" 

# Extract the ZIP file to the base path
zip_file.extractall(base_path)

# Define the paths
paths = {
    "claims.csv": f"{base_path}/claims",
    "diagnoses.csv": f"{base_path}/diagnosis",
    "procedures.csv": f"{base_path}/procedures",
    "member.csv": f"{base_path}/members",
    "providers.csv": f"{base_path}/providers"
}

# Create the destination directories if they do not exist
for dest_path in paths.values():
    os.makedirs(dest_path, exist_ok=True)

# Move the files to the respective directories
for file_name, dest_path in paths.items():
    source_file = f"{base_path}/{file_name}"
    if os.path.exists(source_file):
        os.rename(source_file, f"{dest_path}/{file_name}")


# Copy the files to the specified directories and print the paths
shutil.copy(f"{base_path}/claims/claims.csv", f"/Volumes/{catalog}/{bronze_db}/payer/files/claims/claims.csv")
print(f"Copied to /Volumes/{catalog}/{bronze_db}/payer/files/claims/claims.csv")

shutil.copy(f"{base_path}/diagnosis/diagnoses.csv", f"/Volumes/{catalog}/{bronze_db}/payer/files/diagnosis/diagnosis.csv")
print(f"Copied to /Volumes/{catalog}/{bronze_db}/payer/files/diagnosis/diagnosis.csv")

shutil.copy(f"{base_path}/procedures/procedures.csv", f"/Volumes/{catalog}/{bronze_db}/payer/files/procedures/procedures.csv")
print(f"Copied to /Volumes/{catalog}/{bronze_db}/payer/files/procedures/procedures.csv")

shutil.copy(f"{base_path}/members/member.csv", f"/Volumes/{catalog}/{bronze_db}/payer/files/members/members.csv")
print(f"Copied to /Volumes/{catalog}/{bronze_db}/payer/files/members/members.csv")

shutil.copy(f"{base_path}/providers/providers.csv", f"/Volumes/{catalog}/{bronze_db}/payer/files/providers/providers.csv")
print(f"Copied to /Volumes/{catalog}/{bronze_db}/payer/files/providers/providers.csv")



# 🚀 Let's Build Your First Data Pipeline!

---

## Workshop Roadmap

```
📥 Bronze Layer    →    🔧 Silver Layer    →    ⭐ Gold Layer    →    📊 Analytics
   (Raw Data)          (Cleaned Data)        (Business Tables)      (Insights)
```

In the following sections, we'll build a complete data pipeline following the **Medallion Architecture**:

1. **Bronze Layer**: Ingest raw CSV files into Delta tables
2. **Silver Layer**: Clean, deduplicate, and transform data
3. **Gold Layer**: Create enriched analytics tables
4. **Analytics**: Generate insights and visualizations

Let's get started! 🎉

# 📥 Bronze/Silver Layers – Streamlined Data Preparation

---

## Overview: Simplified Bronze & Silver

For analytics, we'll **streamline** Bronze and Silver layers to quickly get to Gold layer insights:

### Bronze Layer (Raw Data Landing)
- 📂 Ingest encounter data "as-is" using `COPY INTO`
- 💾 Store in Delta Lake for audit trails
- ⏱️ Maintain full history for compliance

### Silver Layer (Clean & Validate)
- 🧹 Remove duplicates and validate data quality
- 🔄 Map ICD-10 codes to HCC categories
- ✅ Apply business rules for CMS submission eligibility

> **💡 Focus**: We'll execute Bronze/Silver steps efficiently so we can spend more time on **Gold layer analytics** that drive business value!

---



## Step 1: Verify Source Files

Let's first check that our source files are available:

In [0]:
%sql
LIST '/Volumes/my_catalog/payer_bronze/payer/files/claims/'

## Step 2: Load Data with COPY INTO

### 📖 Understanding COPY INTO

`COPY INTO` is Databricks' recommended command for loading data from cloud storage into Delta tables.

**Key Benefits:**
- ✅ **Idempotent**: Safely re-run without duplicating data
- ✅ **Incremental**: Only loads new files automatically
- ✅ **Schema Evolution**: Can merge new columns with `mergeSchema` option
- ✅ **Atomic**: Either succeeds completely or rolls back

**Syntax:**
```sql
COPY INTO <table_name>
FROM '<source_path>'
FILEFORMAT = CSV
FORMAT_OPTIONS('header' = 'true', 'inferSchema' = 'true')
COPY_OPTIONS('mergeSchema' = 'true')
```

📚 **Learn More:**
- [COPY INTO Documentation](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/delta-copy-into)
- [COPY INTO Examples](https://learn.microsoft.com/en-us/azure/databricks/ingestion/cloud-object-storage/copy-into/)


### Loading Data with SQL

In [0]:
%sql
-- Load Claims Data into Bronze Table
CREATE TABLE IF NOT EXISTS payer_bronze.claims_raw;
COPY INTO payer_bronze.claims_raw FROM
(SELECT
*
FROM '/Volumes/my_catalog/payer_bronze/payer/files/claims/')
FILEFORMAT = CSV
FORMAT_OPTIONS('header' = 'true',
               'inferSchema' = 'true',
               'delimiter' = ',')
COPY_OPTIONS ('mergeSchema' = 'true', 'force' = 'true');

-- NOTE: 'force = true' is used here for demo purposes only to reload all files every time. In production, omit this option so COPY INTO only processes new data files.


-- Load Diagnosis Data into Bronze Table
CREATE TABLE IF NOT EXISTS payer_bronze.diagnosis_raw;
COPY INTO payer_bronze.diagnosis_raw FROM
(SELECT
*
FROM '/Volumes/my_catalog/payer_bronze/payer/files/diagnosis/')

FILEFORMAT = CSV
FORMAT_OPTIONS('header' = 'true',
               'inferSchema' = 'true',
               'delimiter' = ',')
COPY_OPTIONS ('mergeSchema' = 'true');


-- Load Members Data into Bronze Table
CREATE TABLE IF NOT EXISTS payer_bronze.members_raw;
COPY INTO payer_bronze.members_raw FROM
(SELECT
*
FROM '/Volumes/my_catalog/payer_bronze/payer/files/members/')

FILEFORMAT = CSV
FORMAT_OPTIONS('header' = 'true',
               'inferSchema' = 'true',
               'delimiter' = ',')
COPY_OPTIONS ('mergeSchema' = 'true');


-- Load Procedures Data into Bronze Table
CREATE TABLE IF NOT EXISTS payer_bronze.procedures_raw;
COPY INTO payer_bronze.procedures_raw FROM
(SELECT
*
FROM '/Volumes/my_catalog/payer_bronze/payer/files/procedures/')
FILEFORMAT = CSV
FORMAT_OPTIONS('header' = 'true',
               'inferSchema' = 'true',
               'delimiter' = ',')
COPY_OPTIONS ('mergeSchema' = 'true');


-- Load Providers Data into Bronze Table
CREATE TABLE IF NOT EXISTS payer_bronze.providers_raw;
COPY INTO payer_bronze.providers_raw FROM
(SELECT
*
FROM '/Volumes/my_catalog/payer_bronze/payer/files/providers/')
FILEFORMAT = CSV
FORMAT_OPTIONS('header' = 'true',
               'inferSchema' = 'true',
               'delimiter' = ',')
COPY_OPTIONS ('mergeSchema' = 'true');


### 🐍 Alternative: Loading Data with PySpark

While SQL is great for batch loading, PySpark gives you more programmatic control. Here's how to load the same data using PySpark:

In [0]:
# Example: Load data using PySpark
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType

# Option 1: Let Spark infer the schema
claims_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/my_catalog/payer_bronze/payer/files/claims/")

# Display first 10 rows
display(claims_df.limit(10))

# Show schema
print("Claims Schema:")
claims_df.printSchema()

# Get row count
print(f"\nTotal rows loaded: {claims_df.count()}")

# Write to Delta table (this creates or replaces the table)
# claims_df.write \
#     .format("delta") \
#     .mode("overwrite") \
#     .saveAsTable("payer_bronze.claims_raw_pyspark")


## Silver Layer – Transformation


## Step 1: Transform Bronze to Silver (SQL)

Let's clean and transform our Bronze tables. We'll demonstrate with multiple examples using both **SQL** and **PySpark**.

In [0]:
%sql
-- Create silver schema
CREATE SCHEMA IF NOT EXISTS payer_silver;


-- Members: select relevant fields, cast types, remove duplicates
CREATE OR REPLACE TABLE payer_silver.members AS
SELECT
  DISTINCT CAST(member_id AS STRING) AS member_id,
  TRIM(first_name) AS first_name,
  TRIM(last_name) AS last_name,
  CAST(birth_date AS DATE) AS birth_date,
  gender,
  plan_id,
  CAST(effective_date AS DATE) AS effective_date
FROM payer_bronze.members_raw
WHERE member_id IS NOT NULL;


-- Claims: remove duplicates, prepare data
CREATE OR REPLACE TABLE payer_silver.claims AS
SELECT
  DISTINCT claim_id,
  member_id,
  provider_id,
  CAST(claim_date AS DATE) AS claim_date,
  ROUND(total_charge, 2) AS total_charge,
  LOWER(claim_status) AS claim_status
FROM payer_bronze.claims_raw
WHERE claim_id IS NOT NULL AND total_charge > 0;


-- Providers: deduplicate
CREATE OR REPLACE TABLE payer_silver.providers AS
SELECT
  DISTINCT provider_id,
  npi,
  provider_name,
  specialty,
  address,
  city,
  state
FROM payer_bronze.providers_raw
WHERE provider_id IS NOT NULL;


## Step 2: Transform with PySpark

Now let's see how to do the same transformations using PySpark. This approach is more flexible for complex business logic.

### Example: Transform Procedures Table with PySpark


In [0]:
from pyspark.sql.functions import col, trim, upper, round as spark_round, when, regexp_replace

# Read from Bronze
procedures_bronze = spark.table("payer_bronze.procedures_raw")

# Clean and cast the amount column
procedures_bronze_clean = procedures_bronze.withColumn(
    "amount_clean",
    regexp_replace(col("amount"), "[^0-9.]", "").cast("double")
)

# Apply transformations
procedures_silver = procedures_bronze_clean \
    .dropDuplicates(['claim_id', 'procedure_code']) \
    .filter(col("claim_id").isNotNull()) \
    .filter(col("amount_clean") > 0) \
    .select(
        col("claim_id"),
        upper(trim(col("procedure_code"))).alias("procedure_code"),
        trim(col("procedure_desc")).alias("procedure_desc"),
        spark_round(col("amount_clean"), 2).alias("amount"),
        when(col("amount_clean") < 100, "Low")
        .when(col("amount_clean") < 500, "Medium")
        .when(col("amount_clean") < 1000, "High")
        .otherwise("Very High").alias("cost_category")
    )

# Show sample data
print("Transformed Procedures (first 10 rows):")
display(procedures_silver.limit(10))

# Show statistics
print("\nCost Category Distribution:")
display(procedures_silver.groupBy("cost_category").count().orderBy("cost_category"))

# Write to Silver table
procedures_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("payer_silver.procedures")


# 🤖 Using Databricks AI Assistant

---

Databricks AI Assistant can help you write code, understand data, and troubleshoot issues!

### How to Use AI Assistant:
1. Click the AI Assistant icon
2. Ask questions in natural language
3. Get code suggestions and explanations

### Example Prompts to Try:
- "How do I calculate the total claims by specialty?"
- "Show me how to create a window function for running totals"
- "What does spark.table() command do?"
- "Help me debug this PySpark error"

---



## 🎯 YOUR TURN! Exercise 1 (3 mins)
Ask Databricks Assistant: "How do I calculate the total claims by specialty in SQL?"

In [0]:
%sql
-- YOUR TURN: Write your solution here


## 💡 Solution: Exercise 1

Click below to reveal the solution (try it yourself first!):

In [0]:
%sql
-- Solution
SELECT
    claim_status AS specialty,
    SUM(total_charge) AS total_claims
FROM my_catalog.payer_silver.claims
GROUP BY claim_status

# ⭐ Gold Layer – Operational Analytics & Reporting

---

## What is the Gold Layer?

The **Gold Layer** is where we deliver **business value** for stakeholders. Here we create analytics tables that directly support:

- 📞 **Contact Center Operations**: Call volumes, wait times, agent performance
- ✅ **Compliance Reporting**: State and federal regulatory requirements
- 💰 **Claims Operations**: Processing metrics, turnaround times, backlogs
- 👥 **Member Services & Enrollment**: Open enrollment tracking, member support metrics
- 📈 **Operational Dashboards**: Day-to-day KPIs for business operations
- 🔍 **Research & Insights**: Ad-hoc analysis and trend identification

> **🎯 Business Value**: Each Gold table directly answers operational questions that support daily business operations!

---

## Five SQL Examples for BI Operational Reporting

In the following sections, we'll build **5 SQL-based analytics tables** that reflect your daily work:

1. 📞 **Contact Center Performance Metrics** - Call data and agent performance
2. ✅ **Claims Processing Compliance Report** - Regulatory compliance tracking
3. 👥 **Member Services Enrollment Tracking** - Open enrollment metrics
4. 📊 **Operational Daily Dashboard** - Day-to-day business KPIs
5. 🔍 **Data Quality Audit for Compliance** - Data validation for regulatory reporting

---

## Example 1: Contact Center Performance Metrics

### 📞 Business Goal
Track contact center operational metrics to monitor daily performance and identify areas for improvement.

**Use Case**: As a BI analyst supporting the contact center, you need to create daily/weekly reports showing:
- Call volumes by time period and channel
- Average wait times and handle times
- Agent performance and productivity
- Call resolution rates
- Peak hour identification for staffing

This example demonstrates:
- Multi-table joins with claims and member data
- Aggregations with grouping by time periods
- Window functions for ranking and running totals
- Calculated KPIs for operational metrics

---

### Databricks SQL Solution

In [0]:
%sql
-- Example 1: Contact Center Performance Metrics
-- Simulates call/contact data using claims as a proxy for interactions

CREATE OR REPLACE TABLE payer_gold.contact_center_daily_metrics AS
WITH call_data AS (
  SELECT
    c.claim_id as contact_id,
    c.member_id,
    c.claim_date as contact_date,
    DATE_TRUNC('day', c.claim_date) as contact_day,
    HOUR(c.claim_date) as contact_hour,
    c.claim_status,
    CASE 
      WHEN MOD(ABS(HASH(c.claim_id)), 4) = 0 THEN 'Phone'
      WHEN MOD(ABS(HASH(c.claim_id)), 4) = 1 THEN 'Email'
      WHEN MOD(ABS(HASH(c.claim_id)), 4) = 2 THEN 'Chat'
      ELSE 'Portal'
    END as contact_channel,
    CASE 
      WHEN c.claim_status = 'approved' THEN 'Resolved First Contact'
      WHEN c.claim_status = 'paid' THEN 'Resolved Follow-up'
      ELSE 'Pending Resolution'
    END as resolution_status,
    ROUND(60 + (RAND() * 300), 0) as wait_time_seconds,
    ROUND(180 + (RAND() * 600), 0) as handle_time_seconds
  FROM payer_silver.claims c
  WHERE c.claim_date IS NOT NULL
),
daily_metrics AS (
  SELECT
    contact_day,
    contact_channel,
    COUNT(*) as total_contacts,
    COUNT(DISTINCT member_id) as unique_members,
    ROUND(AVG(wait_time_seconds), 1) as avg_wait_time_seconds,
    ROUND(AVG(handle_time_seconds), 1) as avg_handle_time_seconds,
    ROUND(PERCENTILE(wait_time_seconds, 0.90), 1) as p90_wait_time_seconds,
    SUM(CASE WHEN resolution_status = 'Resolved First Contact' THEN 1 ELSE 0 END) as first_contact_resolution_count,
    ROUND(
      100.0 * SUM(CASE WHEN resolution_status = 'Resolved First Contact' THEN 1 ELSE 0 END) / COUNT(*),
      2
    ) as first_contact_resolution_rate,
    MODE(contact_hour) as peak_hour
  FROM call_data
  GROUP BY contact_day, contact_channel
)
SELECT
  contact_day,
  contact_channel,
  total_contacts,
  unique_members,
  avg_wait_time_seconds,
  avg_handle_time_seconds,
  p90_wait_time_seconds,
  first_contact_resolution_count,
  first_contact_resolution_rate,
  peak_hour,
  ROUND(
    100.0 * SUM(CASE WHEN avg_wait_time_seconds <= 60 THEN total_contacts ELSE 0 END) 
    OVER (PARTITION BY contact_day) / SUM(total_contacts) OVER (PARTITION BY contact_day),
    2
  ) as service_level_60s,
  SUM(total_contacts) OVER (
    PARTITION BY contact_channel 
    ORDER BY contact_day 
    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
  ) as rolling_7day_contacts
FROM daily_metrics
ORDER BY contact_day DESC, contact_channel;


In [0]:
%sql
-- Show the data
SELECT * FROM payer_gold.contact_center_daily_metrics;

### 🎯 Key SQL Features Demonstrated

**This query shows:**
- ✅ **CTEs (Common Table Expressions)**: Clean, modular query structure
- ✅ **Window Functions**: `SUM() OVER()` for running totals and service levels
- ✅ **Advanced Aggregations**: `PERCENTILE()`, `MODE()` for statistical analysis
- ✅ **Date Functions**: `DATE_TRUNC()`, `HOUR()` for time-based grouping
- ✅ **CASE Statements**: Business logic for categorization

**Business Value:**
- 📊 Monitor contact center performance in real-time
- 📈 Identify peak hours for staffing optimization
- ✅ Track SLA compliance (service level agreements)
- 🔍 Analyze resolution rates and handle times by channel

---


## Example 2: Claims Processing Compliance Report

### ✅ Business Goal
Track claims processing metrics to ensure compliance with state and federal requirements for timely processing.

**Use Case**: As a BI analyst supporting claims operations, you need to create compliance reports showing:
- Claims processed within regulatory timeframes (state/federal requirements)
- Processing turnaround time metrics
- Backlog identification and aging analysis
- Compliance rates by claim type and plan
- Trends over time for executive reporting

This example demonstrates:
- Time-based calculations for turnaround metrics
- Compliance threshold validation
- Aging bucket categorization
- Trend analysis with window functions

---

### Databricks SQL Solution

In [0]:
%sql
-- Example 2: Claims Processing Compliance Report

CREATE OR REPLACE TABLE payer_gold.claims_compliance_report AS
WITH claims_with_processing_time AS (
  SELECT
    c.claim_id,
    c.member_id,
    c.claim_date,
    c.claim_status,
    c.total_charge,
    m.plan_id,
    p.provider_id,
    p.state as provider_state,
    CURRENT_DATE() as report_date,
    DATEDIFF(CURRENT_DATE(), c.claim_date) as days_since_claim,
    CASE 
      WHEN c.claim_status IN ('approved', 'paid') THEN DATEDIFF(DATE_ADD(c.claim_date, CAST(RAND() * 45 AS INT)), c.claim_date)
      ELSE DATEDIFF(CURRENT_DATE(), c.claim_date)
    END as processing_days,
    CASE 
      WHEN c.total_charge < 1000 THEN 'Simple'
      WHEN c.total_charge < 5000 THEN 'Standard'
      ELSE 'Complex'
    END as claim_complexity
  FROM payer_silver.claims c
  INNER JOIN payer_silver.members m ON c.member_id = m.member_id
  INNER JOIN payer_silver.providers p ON c.provider_id = p.provider_id
  WHERE c.claim_date IS NOT NULL
),
compliance_analysis AS (
  SELECT
    claim_id,
    member_id,
    plan_id,
    provider_state,
    claim_date,
    claim_status,
    total_charge,
    claim_complexity,
    processing_days,
    days_since_claim,
    CASE 
      WHEN claim_status IN ('approved', 'paid') AND processing_days <= 30 THEN 'Compliant'
      WHEN claim_status IN ('approved', 'paid') AND processing_days > 30 THEN 'Late but Resolved'
      WHEN claim_status = 'pending' AND days_since_claim <= 30 THEN 'Within SLA'
      WHEN claim_status = 'pending' AND days_since_claim > 30 THEN 'SLA Breach'
      ELSE 'Under Review'
    END as compliance_status,
    CASE
      WHEN claim_status = 'pending' AND days_since_claim <= 15 THEN '0-15 days'
      WHEN claim_status = 'pending' AND days_since_claim <= 30 THEN '16-30 days'
      WHEN claim_status = 'pending' AND days_since_claim <= 45 THEN '31-45 days'
      WHEN claim_status = 'pending' AND days_since_claim <= 60 THEN '46-60 days'
      WHEN claim_status = 'pending' THEN '60+ days'
      ELSE 'Resolved'
    END as aging_bucket
  FROM claims_with_processing_time
)
SELECT
  claim_id,
  member_id,
  plan_id,
  provider_state,
  claim_date,
  claim_status,
  claim_complexity,
  processing_days,
  days_since_claim,
  total_charge,
  compliance_status,
  aging_bucket,
  CASE 
    WHEN compliance_status IN ('Compliant', 'Within SLA') THEN 1 
    ELSE 0 
  END as is_compliant
FROM compliance_analysis
ORDER BY days_since_claim DESC, claim_date DESC;

In [0]:
%sql
-- Show the data
SELECT * FROM payer_gold.claims_compliance_report;

In [0]:
%sql
-- TODO: check what is this?
-- Compliance Summary Report by Plan and State

SELECT
  provider_state,
  plan_id,
  claim_complexity,
  COUNT(*) as total_claims,
  SUM(is_compliant) as compliant_claims,
  ROUND(100.0 * SUM(is_compliant) / COUNT(*), 2) as compliance_rate_pct,
  ROUND(AVG(processing_days), 1) as avg_processing_days,
  ROUND(AVG(total_charge), 2) as avg_claim_amount,
  SUM(CASE WHEN aging_bucket = '60+ days' THEN 1 ELSE 0 END) as aged_backlog_count,
  SUM(CASE WHEN compliance_status = 'SLA Breach' THEN 1 ELSE 0 END) as sla_breach_count
FROM payer_gold.claims_compliance_report
GROUP BY provider_state, plan_id, claim_complexity
ORDER BY compliance_rate_pct ASC, aged_backlog_count DESC
LIMIT 20;


## 🎯 YOUR TURN! Exercise 2: Provider Performance by State (10 mins)

### 📋 Business Context
As a BI analyst, you're asked to create a **provider performance report by state** for operational monitoring. This report helps:
- 📊 Identify which states have the highest claims volume
- 💰 Track total claims amount by provider state
- 📈 Monitor average claim amounts for cost analysis
- 🏥 Count active providers per state

### ✍️ Your Task
Create a Gold table called **`payer_gold.provider_state_summary`** that shows:

1. **State-level aggregations** from claims and provider data
2. **Key metrics**:
   - Total number of claims
   - Total claims dollar amount
   - Average claim amount
   - Number of unique providers
   - Number of unique members served
3. **Filter**: Only include states with at least 10 claims
4. **Sort**: Order by total claims amount (descending)

### 💡 Hints:
- Join `payer_silver.claims` with `payer_silver.providers` 
- Group by `provider_state` (from providers table)
- Use `COUNT()`, `SUM()`, `AVG()` aggregations
- Use `HAVING` clause to filter groups
- Use `ROUND()` for currency values

### 📝 Write your SQL query in the cell below!

In [0]:
%sql
-- YOUR TURN: Write your solution here
-- Create the provider_state_summary table


## 💡 Solution: Exercise 2 - Provider Performance by State

Click below to reveal the solution (try it yourself first!):

In [0]:
%sql
-- Solution: Provider Performance by State

CREATE OR REPLACE TABLE payer_gold.provider_state_summary AS
SELECT
  p.state as provider_state,
  COUNT(*) as total_claims,
  COUNT(DISTINCT c.member_id) as unique_members_served,
  COUNT(DISTINCT p.provider_id) as unique_providers,
  SUM(c.total_charge) as total_claims_amount,
  ROUND(AVG(c.total_charge), 2) as avg_claim_amount,
  ROUND(SUM(c.total_charge) / COUNT(DISTINCT p.provider_id), 2) as avg_amount_per_provider,
  ROUND(100.0 * SUM(CASE WHEN c.claim_status = 'approved' THEN 1 ELSE 0 END) / COUNT(*), 2) as approval_rate_pct
FROM payer_silver.claims c
INNER JOIN payer_silver.providers p ON c.provider_id = p.provider_id
WHERE p.state IS NOT NULL
GROUP BY p.state
--HAVING COUNT(*) >= 10
ORDER BY total_claims_amount DESC;

-- View the results
SELECT * FROM payer_gold.provider_state_summary LIMIT 10;

## Example 3: Member Services Enrollment Tracking

### 👥 Business Goal
Track open enrollment metrics and member services activity to support planning and operations.

**Use Case**: As a BI analyst supporting member services, you need to monitor:
- Open enrollment participation rates by plan and demographics
- New enrollments vs renewals vs terminations
- Member services contact volume during enrollment periods
- Plan selection trends and member retention
- Support operational planning for peak enrollment periods

This example demonstrates:
- Enrollment trend analysis over time
- Member segmentation by enrollment type
- Retention rate calculations
- Period-over-period comparisons

---

### Databricks SQL Solution

In [0]:
%sql
-- Example 3: Member Services Enrollment Tracking

CREATE OR REPLACE TABLE payer_gold.enrollment_metrics AS
WITH member_enrollment_history AS (
  SELECT
    m.member_id,
    m.plan_id,
    m.first_name,
    m.last_name,
    m.birth_date,
    m.gender,
    m.effective_date,
    YEAR(m.effective_date) as enrollment_year,
    MONTH(m.effective_date) as enrollment_month,
    DATE_TRUNC('month', m.effective_date) as enrollment_month_date,
    YEAR(CURRENT_DATE()) - YEAR(m.birth_date) as current_age,
    CASE 
      WHEN YEAR(CURRENT_DATE()) - YEAR(m.birth_date) < 30 THEN '18-29'
      WHEN YEAR(CURRENT_DATE()) - YEAR(m.birth_date) < 45 THEN '30-44'
      WHEN YEAR(CURRENT_DATE()) - YEAR(m.birth_date) < 65 THEN '45-64'
      ELSE '65+'
    END as age_group,
    CASE 
      WHEN MONTH(m.effective_date) IN (10, 11, 12) THEN 'Open Enrollment'
      WHEN MONTH(m.effective_date) = 1 THEN 'Post-OE'
      ELSE 'Special Enrollment'
    END as enrollment_period_type,
    DATEDIFF(CURRENT_DATE(), m.effective_date) as days_enrolled,
    COUNT(DISTINCT c.claim_id) as claim_count
  FROM payer_silver.members m
  LEFT JOIN payer_silver.claims c ON m.member_id = c.member_id
  WHERE m.effective_date IS NOT NULL
  GROUP BY 
    m.member_id, m.plan_id, m.first_name, m.last_name, m.birth_date, 
    m.gender, m.effective_date
),
enrollment_status AS (
  SELECT
    *,
    CASE 
      WHEN days_enrolled < 90 THEN 'New'
      WHEN days_enrolled < 365 THEN 'Recent'
      ELSE 'Established'
    END as member_tenure,
    CASE 
      WHEN claim_count = 0 THEN 'No Activity'
      WHEN claim_count < 5 THEN 'Low Activity'
      WHEN claim_count < 15 THEN 'Moderate Activity'
      ELSE 'High Activity'
    END as engagement_level
  FROM member_enrollment_history
)
SELECT
  enrollment_month_date,
  enrollment_year,
  enrollment_month,
  enrollment_period_type,
  plan_id,
  age_group,
  gender,
  member_tenure,
  engagement_level,
  COUNT(DISTINCT member_id) as total_members,
  ROUND(AVG(current_age), 1) as avg_age,
  ROUND(AVG(days_enrolled), 0) as avg_days_enrolled,
  SUM(claim_count) as total_claims,
  ROUND(AVG(claim_count), 2) as avg_claims_per_member,
  COUNT(DISTINCT CASE WHEN claim_count > 0 THEN member_id END) as active_members,
  ROUND(100.0 * COUNT(DISTINCT CASE WHEN claim_count > 0 THEN member_id END) / COUNT(DISTINCT member_id), 2) as activation_rate_pct
FROM enrollment_status
GROUP BY 
  enrollment_month_date, enrollment_year, enrollment_month, 
  enrollment_period_type, plan_id, age_group, gender, 
  member_tenure, engagement_level
ORDER BY enrollment_month_date DESC, plan_id;

In [0]:
%sql
-- Show the data
SELECT * FROM payer_gold.enrollment_metrics;

In [0]:
%sql
-- TODO: Check what is this? 
-- Open Enrollment Period Summary

SELECT
  enrollment_period_type,
  enrollment_year,
  plan_id,
  SUM(total_members) as total_enrolled,
  ROUND(AVG(avg_age), 1) as avg_member_age,
  SUM(total_claims) as total_claims_volume,
  ROUND(AVG(activation_rate_pct), 2) as avg_activation_rate,
  SUM(CASE WHEN member_tenure = 'New' THEN total_members ELSE 0 END) as new_members,
  SUM(CASE WHEN member_tenure = 'Established' THEN total_members ELSE 0 END) as retained_members,
  ROUND(
    100.0 * SUM(CASE WHEN member_tenure = 'Established' THEN total_members ELSE 0 END) / 
    SUM(total_members), 
    2
  ) as retention_rate_pct
FROM payer_gold.enrollment_metrics
GROUP BY enrollment_period_type, enrollment_year, plan_id
ORDER BY enrollment_year DESC, enrollment_period_type, plan_id
LIMIT 20;


## Example 4: Operational Daily Dashboard

### 📊 Business Goal
Create a comprehensive daily operational dashboard that provides key metrics for day-to-day business operations.

**Use Case**: As a BI analyst, you need to provide leadership with a daily snapshot showing:
- Daily claims processing volume and status
- Member services activity and inquiries
- Provider network utilization
- Financial metrics and trends
- Week-over-week and month-over-month comparisons

This example demonstrates:
- Multi-dimensional aggregations
- Period-over-period comparisons using LAG()
- KPI calculations
- Trend indicators

---

### Databricks SQL Solution

In [0]:
%sql
-- Example 4: Operational Daily Dashboard

CREATE OR REPLACE TABLE payer_gold.operational_dashboard AS
WITH daily_claims_metrics AS (
  SELECT
    DATE_TRUNC('day', claim_date) as business_date,
    COUNT(*) as daily_claims_count,
    COUNT(DISTINCT member_id) as unique_members_with_claims,
    COUNT(DISTINCT provider_id) as unique_providers_servicing,
    SUM(total_charge) as daily_claims_amount,
    ROUND(AVG(total_charge), 2) as avg_claim_amount,
    SUM(CASE WHEN claim_status = 'approved' THEN 1 ELSE 0 END) as approved_claims,
    SUM(CASE WHEN claim_status = 'pending' THEN 1 ELSE 0 END) as pending_claims,
    SUM(CASE WHEN claim_status = 'paid' THEN 1 ELSE 0 END) as paid_claims
  FROM payer_silver.claims
  WHERE claim_date IS NOT NULL
  GROUP BY DATE_TRUNC('day', claim_date)
),
daily_member_activity AS (
  SELECT
    DATE_TRUNC('day', claim_date) as business_date,
    COUNT(DISTINCT m.member_id) as total_active_members,
    COUNT(DISTINCT m.plan_id) as plans_with_activity,
    COUNT(DISTINCT CASE WHEN YEAR(CURRENT_DATE()) - YEAR(m.birth_date) >= 65 THEN m.member_id END) as senior_members_active
  FROM payer_silver.claims c
  INNER JOIN payer_silver.members m ON c.member_id = m.member_id
  WHERE c.claim_date IS NOT NULL
  GROUP BY DATE_TRUNC('day', c.claim_date)
),
trend_calculations AS (
  SELECT
    cm.business_date,
    cm.daily_claims_count,
    cm.unique_members_with_claims,
    cm.unique_providers_servicing,
    cm.daily_claims_amount,
    cm.avg_claim_amount,
    cm.approved_claims,
    cm.pending_claims,
    cm.paid_claims,
    ROUND(100.0 * cm.approved_claims / NULLIF(cm.daily_claims_count, 0), 2) as approval_rate_pct,
    ROUND(100.0 * cm.pending_claims / NULLIF(cm.daily_claims_count, 0), 2) as pending_rate_pct,
    ma.total_active_members,
    ma.plans_with_activity,
    ma.senior_members_active,
    LAG(cm.daily_claims_count, 1) OVER (ORDER BY cm.business_date) as prev_day_claims,
    LAG(cm.daily_claims_count, 7) OVER (ORDER BY cm.business_date) as prev_week_claims,
    LAG(cm.daily_claims_amount, 1) OVER (ORDER BY cm.business_date) as prev_day_amount,
    AVG(cm.daily_claims_count) OVER (ORDER BY cm.business_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) as rolling_7day_avg_claims,
    AVG(cm.daily_claims_amount) OVER (ORDER BY cm.business_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) as rolling_7day_avg_amount
  FROM daily_claims_metrics cm
  LEFT JOIN daily_member_activity ma ON cm.business_date = ma.business_date
)
SELECT
  business_date,
  daily_claims_count,
  unique_members_with_claims,
  unique_providers_servicing,
  daily_claims_amount,
  avg_claim_amount,
  approved_claims,
  pending_claims,
  paid_claims,
  approval_rate_pct,
  pending_rate_pct,
  total_active_members,
  plans_with_activity,
  senior_members_active,
  ROUND(rolling_7day_avg_claims, 1) as rolling_7day_avg_claims,
  ROUND(rolling_7day_avg_amount, 2) as rolling_7day_avg_amount,
  ROUND(100.0 * (daily_claims_count - prev_day_claims) / NULLIF(prev_day_claims, 0), 2) as day_over_day_change_pct,
  ROUND(100.0 * (daily_claims_count - prev_week_claims) / NULLIF(prev_week_claims, 0), 2) as week_over_week_change_pct,
  CASE 
    WHEN daily_claims_count > rolling_7day_avg_claims * 1.2 THEN 'Above Average'
    WHEN daily_claims_count < rolling_7day_avg_claims * 0.8 THEN 'Below Average'
    ELSE 'Normal'
  END as volume_indicator
FROM trend_calculations
ORDER BY business_date DESC;


In [0]:
%sql
-- Show the data
SELECT * FROM payer_gold.operational_dashboard;

## 🎯 YOUR TURN! Exercise 3: Weekly Claims Trend Analysis (10 mins)

### 📋 Business Context
As a BI analyst, you need to create a **weekly claims trend report** for operations and compliance teams. This helps:
- 📈 Track claims volume week-over-week
- 💰 Monitor weekly revenue trends
- 🔍 Identify unusual patterns or anomalies
- 📊 Support capacity planning and staffing decisions

### ✍️ Your Task
Create a Gold table called **`payer_gold.weekly_claims_trends`** that shows:

1. **Weekly aggregations** of claims data
2. **Key metrics per week**:
   - Week start date (Monday of each week)
   - Total claims count
   - Total claims amount
   - Average claim amount
   - Unique members with claims
   - Unique providers servicing claims
3. **Trend indicators**:
   - Previous week's claims count (using `LAG()`)
   - Week-over-week change percentage
   - 4-week rolling average of claims count
4. **Sort**: Order by week start date (most recent first)

### 💡 Hints:
- Use `DATE_TRUNC('week', claim_date)` to get week start dates
- Use `LAG()` window function for previous week comparison
- Use `AVG() OVER (ORDER BY ... ROWS BETWEEN 3 PRECEDING AND CURRENT ROW)` for rolling average
- Calculate percentage change: `(current - previous) * 100.0 / previous`
- Remember to handle NULL values with `COALESCE()` or `NULLIF()`

### 📝 Write your SQL query in the cell below!

In [0]:
%sql
-- YOUR TURN: Write your solution here
-- Create the weekly_claims_trends table



## 💡 Solution: Exercise 3 - Weekly Claims Trend Analysis

Click below to reveal the solution (try it yourself first!):

In [0]:
%sql
-- Solution: Weekly Claims Trend Analysis

CREATE OR REPLACE TABLE payer_gold.weekly_claims_trends AS
WITH weekly_aggregates AS (
  SELECT
    DATE_TRUNC('week', claim_date) as week_start_date,
    COUNT(*) as weekly_claims_count,
    SUM(total_charge) as weekly_claims_amount,
    ROUND(AVG(total_charge), 2) as avg_claim_amount,
    COUNT(DISTINCT member_id) as unique_members,
    COUNT(DISTINCT provider_id) as unique_providers
  FROM payer_silver.claims
  WHERE claim_date IS NOT NULL
  GROUP BY DATE_TRUNC('week', claim_date)
),
trend_calculations AS (
  SELECT
    week_start_date,
    weekly_claims_count,
    weekly_claims_amount,
    avg_claim_amount,
    unique_members,
    unique_providers,
    LAG(weekly_claims_count, 1) OVER (ORDER BY week_start_date) as prev_week_claims_count,
    AVG(weekly_claims_count) OVER (
      ORDER BY week_start_date 
      ROWS BETWEEN 3 PRECEDING AND CURRENT ROW
    ) as rolling_4week_avg_claims
  FROM weekly_aggregates
)
SELECT
  week_start_date,
  weekly_claims_count,
  weekly_claims_amount,
  avg_claim_amount,
  unique_members,
  unique_providers,
  prev_week_claims_count,
  ROUND(rolling_4week_avg_claims, 1) as rolling_4week_avg_claims,
  ROUND(
    100.0 * (weekly_claims_count - prev_week_claims_count) / NULLIF(prev_week_claims_count, 0),
    2
  ) as week_over_week_change_pct,
  CASE
    WHEN weekly_claims_count > rolling_4week_avg_claims * 1.15 THEN 'Above Trend'
    WHEN weekly_claims_count < rolling_4week_avg_claims * 0.85 THEN 'Below Trend'
    ELSE 'Normal'
  END as trend_indicator
FROM trend_calculations
ORDER BY week_start_date DESC;

-- View the results
SELECT * FROM payer_gold.weekly_claims_trends LIMIT 10;

## Example 5: Data Quality Audit for Compliance

### 🔍 Business Goal
Ensure data quality and completeness to support accurate reporting and regulatory compliance.

**Use Case**: As a BI analyst, you need to validate data quality for operational reporting and compliance:
- Identify missing or invalid data elements
- Track data completeness rates
- Monitor data quality trends over time
- Flag records that don't meet compliance standards
- Support state and federal reporting requirements

This example demonstrates:
- Data quality validation rules
- Completeness metrics
- Trend analysis for data quality
- Categorization of data issues

---

### Databricks SQL Solution

In [0]:
%sql
-- Example 5: Data Quality Audit for Compliance

CREATE OR REPLACE TABLE payer_gold.data_quality_audit AS
WITH member_quality_checks AS (
  SELECT
    'Member Data' as data_domain,
    'Total Members' as check_name,
    COUNT(*) as record_count,
    COUNT(*) as valid_count,
    0 as invalid_count,
    100.0 as quality_score_pct,
    'INFO' as severity,
    'Baseline count' as issue_description
  FROM payer_silver.members

  UNION ALL

  SELECT
    'Member Data',
    'Missing Member ID',
    COUNT(*),
    COUNT(*) - COUNT(CASE WHEN member_id IS NULL THEN 1 END),
    COUNT(CASE WHEN member_id IS NULL THEN 1 END),
    ROUND(100.0 * (COUNT(*) - COUNT(CASE WHEN member_id IS NULL THEN 1 END)) / NULLIF(COUNT(*), 0), 2),
    'CRITICAL',
    'Member records without ID'
  FROM payer_silver.members

  UNION ALL

  SELECT
    'Member Data',
    'Invalid Birth Date',
    COUNT(*),
    COUNT(CASE WHEN birth_date IS NOT NULL AND birth_date < '1900-01-01' THEN NULL ELSE 1 END),
    COUNT(CASE WHEN birth_date IS NULL OR birth_date < '1900-01-01' THEN 1 END),
    ROUND(100.0 * COUNT(CASE WHEN birth_date IS NOT NULL AND birth_date >= '1900-01-01' THEN 1 END) / NULLIF(COUNT(*), 0), 2),
    'ERROR',
    'Missing or unrealistic birth dates'
  FROM payer_silver.members

  UNION ALL

  SELECT
    'Member Data',
    'Missing Plan Assignment',
    COUNT(*),
    COUNT(CASE WHEN plan_id IS NOT NULL THEN 1 END),
    COUNT(CASE WHEN plan_id IS NULL THEN 1 END),
    ROUND(100.0 * COUNT(CASE WHEN plan_id IS NOT NULL THEN 1 END) / NULLIF(COUNT(*), 0), 2),
    'ERROR',
    'Members without plan assignment'
  FROM payer_silver.members
),
claims_quality_checks AS (
  SELECT
    'Claims Data' as data_domain,
    'Total Claims' as check_name,
    COUNT(*) as record_count,
    COUNT(*) as valid_count,
    0 as invalid_count,
    100.0 as quality_score_pct,
    'INFO' as severity,
    'Baseline count' as issue_description
  FROM payer_silver.claims

  UNION ALL

  SELECT
    'Claims Data',
    'Missing Claim Date',
    COUNT(*),
    COUNT(CASE WHEN claim_date IS NOT NULL THEN 1 END),
    COUNT(CASE WHEN claim_date IS NULL THEN 1 END),
    ROUND(100.0 * COUNT(CASE WHEN claim_date IS NOT NULL THEN 1 END) / NULLIF(COUNT(*), 0), 2),
    'CRITICAL',
    'Claims without service date'
  FROM payer_silver.claims

  UNION ALL

  SELECT
    'Claims Data',
    'Invalid Claim Amount',
    COUNT(*),
    COUNT(CASE WHEN total_charge > 0 THEN 1 END),
    COUNT(CASE WHEN total_charge IS NULL OR total_charge <= 0 THEN 1 END),
    ROUND(100.0 * COUNT(CASE WHEN total_charge > 0 THEN 1 END) / NULLIF(COUNT(*), 0), 2),
    'ERROR',
    'Claims with zero or negative amounts'
  FROM payer_silver.claims

  UNION ALL

  SELECT
    'Claims Data',
    'Orphaned Claims (No Member)',
    COUNT(*),
    COUNT(m.member_id),
    COUNT(*) - COUNT(m.member_id),
    ROUND(100.0 * COUNT(m.member_id) / NULLIF(COUNT(*), 0), 2),
    'CRITICAL',
    'Claims without matching member'
  FROM payer_silver.claims c
  LEFT JOIN payer_silver.members m ON c.member_id = m.member_id

  UNION ALL

  SELECT
    'Claims Data',
    'Orphaned Claims (No Provider)',
    COUNT(*),
    COUNT(p.provider_id),
    COUNT(*) - COUNT(p.provider_id),
    ROUND(100.0 * COUNT(p.provider_id) / NULLIF(COUNT(*), 0), 2),
    'ERROR',
    'Claims without matching provider'
  FROM payer_silver.claims c
  LEFT JOIN payer_silver.providers p ON c.provider_id = p.provider_id
),
provider_quality_checks AS (
  SELECT
    'Provider Data' as data_domain,
    'Total Providers' as check_name,
    COUNT(*) as record_count,
    COUNT(*) as valid_count,
    0 as invalid_count,
    100.0 as quality_score_pct,
    'INFO' as severity,
    'Baseline count' as issue_description
  FROM payer_silver.providers

  UNION ALL

  SELECT
    'Provider Data',
    'Missing NPI',
    COUNT(*),
    COUNT(CASE WHEN npi IS NOT NULL AND TRIM(npi) != '' THEN 1 END),
    COUNT(CASE WHEN npi IS NULL OR TRIM(npi) = '' THEN 1 END),
    ROUND(100.0 * COUNT(CASE WHEN npi IS NOT NULL AND TRIM(npi) != '' THEN 1 END) / NULLIF(COUNT(*), 0), 2),
    'CRITICAL',
    'Providers without NPI'
  FROM payer_silver.providers
),
all_quality_checks AS (
  SELECT * FROM member_quality_checks
  UNION ALL
  SELECT * FROM claims_quality_checks
  UNION ALL
  SELECT * FROM provider_quality_checks
)
SELECT
  data_domain,
  check_name,
  record_count,
  valid_count,
  invalid_count,
  quality_score_pct,
  severity,
  issue_description,
  CASE 
    WHEN severity = 'CRITICAL' AND invalid_count > 0 THEN 'FAIL'
    WHEN severity = 'ERROR' AND quality_score_pct < 95 THEN 'FAIL'
    WHEN severity = 'ERROR' AND quality_score_pct < 98 THEN 'REVIEW'
    WHEN quality_score_pct < 99 THEN 'REVIEW'
    ELSE 'PASS'
  END as audit_status,
  CURRENT_TIMESTAMP() as audit_run_date
FROM all_quality_checks
ORDER BY 
  CASE severity 
    WHEN 'CRITICAL' THEN 1 
    WHEN 'ERROR' THEN 2 
    WHEN 'WARNING' THEN 3 
    ELSE 4 
  END,
  quality_score_pct ASC;

In [0]:
%sql
-- Show the data
SELECT * FROM payer_gold.data_quality_audit;

In [0]:
%sql
-- TODO: check what is this
-- View Data Quality Audit Results

SELECT 
  data_domain,
  check_name,
  record_count,
  valid_count,
  invalid_count,
  quality_score_pct,
  severity,
  audit_status,
  issue_description
FROM payer_gold.data_quality_audit
WHERE audit_status IN ('FAIL', 'REVIEW')
ORDER BY 
  CASE severity WHEN 'CRITICAL' THEN 1 WHEN 'ERROR' THEN 2 ELSE 3 END,
  quality_score_pct ASC;

---

# AI/BI

Intelligent analytics for everyone!

Databricks AI/BI is a new type of business intelligence product designed to provide a deep understanding of your data's semantics, enabling self-service data analysis for everyone in your organization. AI/BI is built on a compound AI system that draws insights from the full lifecycle of your data across the Databricks platform, including ETL pipelines, lineage, and other queries.

<img src="https://www.databricks.com/sites/default/files/2025-05/hero-image-ai-bi-v2-2x.png?v=1748417271" alt="Managed Tables" width="600" height="500">

# Genie

Talk with your data!

Now everyone can get insights from data simply by asking questions in natural language.

<img src="https://www.databricks.com/sites/default/files/2025-06/ai-bi-genie-hero.png?v=1749162682" alt="Managed Tables" width="600" height="500">


# 🎓 Workshop Summary & Next Steps

## 🎉 Congratulations, BI Organization!

You've completed the **Databricks Data & Analytics Workshop** designed for the **BI Organization**! 

---

## 📋 What You Learned

### ✅ Five SQL Examples for Operational Reporting
1. **Contact Center Performance Metrics** - Monitor call volumes and service levels
2. **Claims Processing Compliance** - Track regulatory compliance and processing times
3. **Enrollment Tracking** - Analyze open enrollment and member retention
4. **Operational Dashboard** - Daily KPIs with trend analysis
5. **Data Quality Audit** - Validate data for compliance reporting

### 🔑 Key Databricks SQL Features
- **CTEs (WITH clause)** - Modular, maintainable queries
- **Window Functions** - `LAG()`, `SUM() OVER()`, `ROW_NUMBER()` for trends
- **Advanced Aggregations** - `PERCENTILE()`, `MODE()` for statistical analysis
- **Date Functions** - `DATE_TRUNC()`, `DATEDIFF()` for time-based analysis
- **UNION ALL** - Combine multiple data quality checks

---

## 🚀 Next Steps for Your Team

### 🛠️ **Apply to Your Work**
1. **Start Small**: Recreate one of your current SQL Server reports in Databricks
2. **Build Dashboards**: Use Databricks SQL for operational dashboards
3. **Share Notebooks**: Collaborate with team members
4. **Prepare for Cloud**: Get ready for upstream data migration to cloud
5. **Automate Reports**: Schedule recurring jobs for daily/weekly reports

### 📖 **Resources**
- [Databricks SQL Reference](https://docs.databricks.com/sql/language-manual/index.html)
- [Delta Lake Guide](https://docs.databricks.com/delta/index.html)
- [Unity Catalog](https://docs.databricks.com/data-governance/unity-catalog/index.html)
- [Databricks SQL Dashboards](https://docs.databricks.com/sql/user/dashboards/index.html)
- **Best Practices Notebook**: _[Reference] Best Practices_

### 💡 **Tips for Success**
- ✅ **Use AI Assistant** - Get help with SQL queries and syntax
- ✅ **Experiment** - Try different approaches, test queries
- ✅ **Collaborate** - Share queries and insights with teammates
- ✅ **Document** - Add comments to your queries for future reference
- ✅ **Think Cloud-Ready** - Prepare for upstream data in Databricks


---

## 🙏 Thank You!

Thank you for participating in this workshop! We hope you found it valuable for your operational reporting needs.

**Welcome to modern cloud analytics with Databricks!** 🚀

---
